# 06 · 编排：让流水线每个月自己跑

## 这一章在整个项目里的位置

```
00–05 ✅ → [06 自动化] → 07–09 训练模型、预测全蛋白组 → 10 对外 API
```

## 背景：为什么需要这一章

到 05 号为止，每一步都是**你亲手按的按钮**：03b 叫送货员进货，04 启动 Databricks 的 Job，05 在电脑上跑 dbt。

但数据会更新：ClinVar 每个月出一版新的。总不能每个月都靠人记着去按。公司里的做法叫**编排**（orchestration）：把步骤连成一张流程图，交给一个**调度员**按时间去按；失败了自动重试，还不行就通知人。

这一章做完以后，每个月会自动发生这些事：

```
每月 10 号 06:00（UTC）  Airflow 自己醒来
   │
   ├─1─ 看 ClinVar 有没有新的月度版本；有就让送货员搬进 S3 raw        （AWS）
   ├─2─ 给 Databricks 一份提货单（临时下载链接）                       （AWS → Databricks）
   └─3─ 启动 Databricks 的流水线 "qafi training data"，等它跑完：      （Databricks）
          搬文件 → bronze → silver → 质检 → dbt 做 gold（05 号的成品表）
```

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 3 步：Terraform 在 AWS 建一个保险柜
   ├─ 第 4 步：在 Databricks 给 Airflow 办一张卡，卡的密码直接进保险柜
   ├─ 第 5 步：把新图纸部署到 Databricks
   └─ 第 6–8 步：在你电脑上启动 Airflow，打开它的网页 http://localhost:8080，看它跑
看结果：Airflow 网页、Databricks 网页（Jobs）、AWS 网页（Secrets Manager，可选）
```

## 这一章做的 10 件事

| # | 做什么 | 用的技术 | 比喻 |
|---|---|---|---|
| 1 | 编排是什么 | 流程图（DAG）、定时、重试 | 实验室的排班表 |
| 2 | 两个调度员，各管一段 | Lakeflow Jobs + Airflow | 仪器上的自动程序 + 排班员 |
| 3 | 保险柜 | AWS Secrets Manager + Terraform | 保险柜 |
| 4 | Airflow 的门禁卡 | 机器身份（service principal）+ OAuth | 只开两扇门的门禁卡 |
| 5 | 把 04 和 05 连成一条流水线 | Lakeflow Jobs、Asset Bundles | 仪器上设好的程序 |
| 6 | 请 Airflow 上岗 | Airflow | 排班员上班 |
| 7 | 读排班表 | DAG 文件 | 排班表 |
| 8 | 试运行一次 | 开关、Trigger、日志 | 第一次带着跑 |
| 9 | 新版本怎么自动进来 | `sources.toml` 的 `refresh` | 订阅期刊 |
| 10 | 关掉 Airflow，提交 | Git、CI | |

**花费**：保险柜每月约 0.4 美元，从 AWS 的免费额度里扣；其他都是 0。**时间**：大约 20 分钟，其中 6 分钟是等流水线跑完。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 Databricks 和 AWS。**不用看代码。**

In [ ]:
import os, subprocess
from datetime import date
from pathlib import Path
import boto3
from databricks.sdk import WorkspaceClient
from qafi_platform import ingest, lakehouse, orchestration
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"   # Terraform、Airflow 和 AWS 调用都用 00 号登录的身份
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
CATALOG, JOB, DAG = "qafi_dev", "qafi training data", "qafi_refresh"
SECRET = "airflow/connections/databricks_default"
print("Databricks:", w.config.host)
print("AWS:", aws.client("sts").get_caller_identity()["Arn"])

> 如果 AWS 报错（`session has expired`）：在终端运行 `aws login --profile qafi`；如果 Databricks 报错：在终端运行 `databricks auth login --profile qafi`。然后再运行这一格。

---

## 第 1 步 · 编排是什么

### 在干什么
认识"编排"：让一串步骤按时、按顺序、自己跑。

### 背景
比喻：**实验室的排班表**。上面写着每周几、几点、谁、按什么顺序做什么；前一步没做完，后一步不开始；仪器出了小故障，等一会儿再试；还不行，就打电话叫负责人。

编排里的几个词，都对应排班表上的东西：

- **流程图（DAG）**：方框是任务，箭头是先后顺序。箭头只往前走，不会绕回来，所以叫"有向无环图"。04 号的 Job 和这一章的 Airflow，画出来都是这样的流程图。
- **定时（schedule）**：像闹钟。我们定的是"每月 10 号 06:00（UTC）"，西班牙夏令时是早上 8 点。
- **重试（retry）**：网络抖一下造成的失败，等一会儿再试就好了。但**质检不过是数据有问题，重试也没用**，所以质检不重试。
- **通知（alert）**：重试也不行，就发邮件叫人。
- **重跑安全（idempotent）**：同一个任务跑两次，结果和跑一次一样。我们的每一步都是这样：已经在的文件跳过，表整张重建。所以失败了直接重跑，不怕"跑了一半"留下的烂摊子。

### 公司里什么情况怎么做
编排是数据工程师每天都要打交道的东西。有人问"今天的数据怎么没更新？"，第一件事就是去调度系统里看哪一步红了。

### 面试怎么说
> "Orchestration turns pipeline steps into a DAG with a schedule, retries and alerts. Because every task is idempotent, any failed run can simply be run again."

---

## 第 2 步 · 两个调度员，各管一段

### 在干什么
弄清楚为什么用两个工具，各管哪一段。

### 背景
- **Lakeflow Jobs**（Databricks 自带）：比喻是**一台自动化工作站上设好的程序**。它只管工作站里面的事：搬文件 → bronze → silver → 质检 → gold。04 号你已经用过它。
- **Airflow**（各公司通用）：比喻是**实验室的排班员**。它自己不做实验，只是到点去各台仪器按按钮：先去 AWS 那边看有没有新数据，再去 Databricks 按"开始"。

为什么不只用一个：
- **只用 Databricks**：它够不着 AWS 那边。免费版连不上我们的 S3，也不应该在 Databricks 里存 AWS 的密码。
- **只用 Airflow**：它也能一步一步启动 Databricks 里的每个任务，但每一步的重试、日志、计算资源，Databricks 自己管得更好。

所以：**Airflow 管跨系统的大顺序，Lakeflow Jobs 管 Databricks 里面的小顺序。**

**一条流水线只设一个闹钟**：Databricks 的 Job 自己也能定时，但我们不设，只让 Airflow 按。两个闹钟都响，同一条流水线就会跑两遍。

### 用了什么技术，为什么
**Airflow** 是最常用的开源调度系统，最早是 Airbnb 写的，现在是 Apache 的项目，几乎每个数据团队都在用。公司里一般用托管版：AWS 上叫 **MWAA**，还有 **Astronomer** 和 Google 的 **Cloud Composer**。它们背后都是同一个 Airflow，流程图的写法一样。

### 公司里什么情况怎么做
- 数据全在 Databricks 里的公司，常常只用 Lakeflow Jobs
- 要串起很多系统的公司，用 Airflow 统一调度

面试时两种都要能讲清楚。

### 面试怎么说
> "Lakeflow Jobs orchestrates the steps inside Databricks; Airflow is the cross-system orchestrator: it brings new data into AWS, then triggers the Databricks job. Only one of them owns the schedule, so the pipeline never runs twice."

---

## 第 3 步 · 保险柜：AWS Secrets Manager

### 在干什么
在 AWS 建一个保险柜，用来放 Airflow 进 Databricks 用的密码。

### 背景
Airflow 每个月自己去 Databricks 按按钮，那时你不在电脑旁。所以它需要自己的密码，也就是第 4 步要办的那张卡的密码。

密码放在哪里？**这些地方绝对不能放**：代码、Git、notebook 的输出、`.env` 文件、聊天记录。这些地方迟早会被别人看到。02 号的 pre-commit 挡的就是这种事。

公司里的答案是**保险柜**。

### 用了什么技术，为什么
**AWS Secrets Manager**：
- **锁**：用 01 号建的那把 KMS 钥匙加密
- **谁能开**：由 IAM 权限决定。现在只有你，以及用你的身份运行的 Airflow
- **谁开过**：CloudTrail 记下每一次读取
- **按名字取**：Airflow 有个约定，要用连接 `<名字>` 时，就去保险柜取 `airflow/connections/<名字>`

**Terraform 只建空柜子，不放密码**：密码如果经过 Terraform，就会被写进 `terraform.tfstate`（状态文件），而状态文件里是明文。所以柜子用 Terraform 建，这符合我们"AWS 资源只由 Terraform 创建"的规则；密码在第 4 步直接从 Databricks 放进去。

### 公司里什么情况怎么做
- 每个系统的密码都放在保险柜里。AWS Secrets Manager、HashiCorp Vault、Azure Key Vault 都是同一个东西
- 程序运行时去取，不写到硬盘上
- 定期换密码，这叫**轮换**

### 执行
先看 Terraform 打算做什么：

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

### 结果怎么看
- `aws_secretsmanager_secret.airflow_databricks will be created`
- `Plan: 1 to add, 0 to change, 0 to destroy.`：只新建 1 个保险柜，别的都不动

没问题就照这个计划建：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")

### 结果怎么看
`Apply complete! Resources: 1 added`：保险柜建好了，现在里面是空的。

### 面试怎么说
> "Credentials live in AWS Secrets Manager, encrypted with our KMS key and audited by CloudTrail. Terraform creates the secret but never its value, so nothing sensitive reaches the state file."

---

## 第 4 步 · Airflow 的门禁卡：机器身份

### 在干什么
在 Databricks 里给 Airflow 办一张它自己的卡，卡的密码直接放进保险柜。

### 背景
比喻：你不会把自己家的钥匙交给快递员，而是给他一张**只能开收货间的门禁卡**。

**为什么不用你自己的账号**：
- 你的账号是管理员，什么都能做，权限太大
- 你一换密码、一离开项目，Airflow 就断了
- 日志里分不清哪件事是你做的，哪件是 Airflow 做的

**机器身份**（service principal）：Databricks 里专门给程序用的账号，不会有人用它登录网页。

**这张卡只开两扇门**，写在第 5 步的图纸里：
1. 启动 `qafi training data` 这条流水线，并查看它跑到哪了
2. 往 raw 文件区放提货单

**密码有有效期：90 天**。到期前把这一格再运行一次，就换一张新卡：新密码进保险柜，旧密码作废。这就是**轮换**。

**有一点要知道**：流水线运行时用的是它主人的身份，也就是你的身份。所以拿到这张卡的人，可以让这条流水线以你的身份跑一次（只能跑，改不了里面的代码）。卡锁在你自己的保险柜里，这个风险可以接受。公司里会再进一步：流水线由 CI 用机器身份部署，运行时也用机器身份。

### 用了什么技术，为什么
**OAuth 机器对机器登录**（OAuth M2M）：Airflow 每次用"卡号 + 密码"换一张**一小时的通行证**，再拿通行证去调用 Databricks。这和 02 号 GitHub 用 OIDC 进 AWS 是同一个思路：真正干活用的，都是短期通行证。

### 公司里什么情况怎么做
- 每个自动化程序（Airflow、CI、其他服务）各有自己的机器身份
- 权限只给够用的，这叫**最小权限**
- 有人离职、换人，都不影响这些程序

### 执行
第一行：有卡就换新密码，没有就新建一张；密码从 Databricks 直接进保险柜，屏幕上不显示。第二行：让这张卡能进 `qafi_dev` 这栋楼。04 号说过，catalog 是楼、schema 是楼层。这栋楼是 04 号手工盖的，不在图纸里，所以楼门用一句 SQL 打开；楼层和房间的门在第 5 步的图纸里。

In [ ]:
APP_ID = lakehouse.create_machine_identity(w, aws.client("secretsmanager"), "qafi-airflow", SECRET)
sql(w, f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{APP_ID}`")
print("qafi-airflow 的卡号:", APP_ID)

### 结果怎么看
打印出的一串字母和数字是**卡号**（application ID），不是密码。密码没有出现在任何地方，它从 Databricks 直接进了保险柜。

看一眼保险柜的情况（只看名字、用哪把钥匙锁、什么时候放进去的，不看密码本身）：

In [ ]:
{k: v for k, v in aws.client("secretsmanager").describe_secret(SecretId=SECRET).items() if k in ("Name", "KmsKeyId", "LastChangedDate")}

### 去网页上看（可选）
- **Databricks**：右上角头像 → Settings → Identity and access → Service principals → `qafi-airflow`
- **AWS**：搜索 Secrets Manager → `airflow/connections/databricks_default`。**不要点 Retrieve secret value**，那会把密码显示在屏幕上

### 面试怎么说
> "Airflow authenticates to Databricks as a service principal with OAuth machine-to-machine. Its secret goes straight into Secrets Manager, rotates every 90 days, and the identity can only start one job and drop the transfer list."

---

## 第 5 步 · 一条流水线：把 04 和 05 连起来

### 在干什么
让 Databricks 的 Job 把 05 号的 dbt 也一起做了，一口气从 raw 做到 gold。

### 背景
04 号的 Job 做到 silver 和质检就停了；05 号的 dbt 是你在电脑上手动跑的。现在把它们连成一条流水线：

```
copy_raw ─┬─▶ bronze_reference ─┬─▶ silver_dms ─▶ quality_checks ─▶ gold（dbt build）
          └─▶ bronze_dms ────────┘
```

还记得 04 号的比喻吗：图纸（Asset Bundle）是**实验方案**，写大的框架；每个 `.py` 是一步的实验细节。这一章只改了方案，实验细节一个都没改。

### 用了什么技术，为什么
图纸 [`databricks/resources/lakehouse.yml`](../../databricks/resources/lakehouse.yml) 里改了这些：

| 改了什么 | 作用 | 为什么 |
|---|---|---|
| 加了 `gold` 这一步（dbt 任务） | 在 Databricks 里跑 05 号的 `dbt build` | Databricks 自己生成 dbt 的连接信息，不需要令牌；部署时 `dbt/` 文件夹会一起带上去 |
| 版本参数 | 每次运行都写明用的是哪一版 ClinVar、UniProt、ProteinGym | 可追溯：以后能查到某次结果用的是哪批数据 |
| `copy_raw` 失败自动重试 2 次 | 网络抖一下不算失败 | 质检不重试：数据错了，重试也没用 |
| 最多跑 1 小时 | 卡住了自动停 | 不会白白耗光免费额度 |
| 失败就发邮件 | 发给你 | 邮箱在部署时自动填进去，不写进 Git |
| 权限表 | qafi-airflow 能启动、能查看 | 第 4 步说的第一扇门 |
| raw 楼层和文件区的门 | qafi-airflow 能进 raw、能放提货单 | 第二扇门 |

### 公司里什么情况怎么做
Job 和权限都写在图纸里，改的时候走 Pull Request。**没有人在网页上手动改权限**：手动改的东西下次部署可能被覆盖，而且没有记录。

### 执行

In [ ]:
run("cd databricks && databricks bundle validate && databricks bundle deploy")

### 结果怎么看
- `Validation OK!`
- `Deployment complete!`

### 去网页上看
Databricks → **Jobs & Pipelines** → `qafi training data`：
- **Tasks**：6 个方框，最后一个是 `gold`（07b/07c 又加了 MaveDB 两步，所以你现在看到的是 8 个）
- 右上角的 **Permissions**：`qafi-airflow` 是 **Can Manage Run**（能启动、能查看，不能修改）

### 面试怎么说
> "One Lakeflow job runs ingestion, quality checks and the dbt build on serverless compute. Each run takes the dataset versions as parameters; the download step retries while quality failures do not, and a failure emails the owner."

---

## 第 6 步 · 请 Airflow 上岗

### 在干什么
在你的电脑上启动 Airflow，打开它的网页。

### 背景
Airflow 由几部分组成：
- **调度器**（scheduler）：盯着时钟和流程图，时间一到就把任务排上队
- **网页**（API server）：你看进度、看日志、按按钮的地方
- **执行者**（worker）：真正运行每个任务的程序
- **数据库**：记住每一次运行、每个任务的状态

`airflow standalone` 是官方的"本机一键版"：一条命令把这几部分全部启动，适合学习和开发。

### 用了什么技术，为什么
**Airflow 3.3**，2026 年 9 月发布的版本。它装在一个**单独的环境**里（[`airflow/pyproject.toml`](../../airflow/pyproject.toml)）：Airflow 依赖几百个包，不能让它来决定主项目用什么版本。

启动脚本 [`airflow/airflow.sh`](../../airflow/airflow.sh) 统一设置了这些：
- 数据库和日志放在 `airflow/.airflow`，Git 不收
- 连接去 AWS Secrets Manager 取，也就是第 3 步的保险柜
- 网页只有你这台电脑能打开（127.0.0.1），同一个 Wi-Fi 里的别人打不开
- 访问 AWS 用你 `aws login` 的身份

### 公司里什么情况怎么做
公司里的 Airflow 是 **24 小时开着的托管服务**（AWS 的 MWAA、Astronomer）：
- DAG 文件由 CI 从 Git 自动部署上去
- Airflow 用自己的 IAM 角色访问 AWS，不用某个人的登录

我们本机这套是同一个 Airflow，区别只是电脑一关，它就不跑了。

### 执行

In [ ]:
airflow = orchestration.start_airflow(ROOT)
print("打开 http://localhost:8080   用户名 admin   密码", orchestration.airflow_password(ROOT))

### 结果怎么看
半分钟左右，打印出网址和密码。这个密码只在你电脑上有用。

### 在网页上
1. 打开 http://localhost:8080 ，用 `admin` 和上面的密码登录
2. 想看中文：左下角的用户图标里，可以把语言换成简体中文
3. 点 **Dags** → `qafi_refresh` → **Graph**：3 个方框连成一条线
4. 这时它是**暂停**的：新上线的流程图默认暂停，防止一上线就乱跑

### 面试怎么说
> "Locally I run Airflow standalone with the configuration a managed Airflow would have: DAGs from git, connections from Secrets Manager. In production it would be MWAA or Astronomer, with its own IAM role."

---

## 第 7 步 · 读排班表：DAG 文件

### 在干什么
看懂 [`airflow/dags/qafi_refresh.py`](../../airflow/dags/qafi_refresh.py) 在说什么。

### 背景
DAG 文件就是**排班表**，用 Python 写。它回答四个问题：

**1. 什么时候跑**：`schedule="0 6 10 * *"`，每月 10 号 06:00（UTC）。为什么选 10 号：ClinVar 每月初发布新版，到 10 号一定已经出来了。

**2. 跑哪几步**：3 个任务
- `fetch_new_releases`：看 `sources.toml` 里标了"每月更新"的数据（现在只有 ClinVar）有没有新版本，有就让 03b 的送货员搬进 S3 raw；已经在的跳过
- `send_transfer_list`：用机器身份给 Databricks 放提货单，也就是 04 号用过的那种临时下载链接
- `run_lakehouse_job`：用机器身份启动第 5 步的流水线，把第一步找到的版本号传过去，然后每 30 秒看一眼，直到跑完

**3. 按什么顺序**：1 → 2 → 3

**4. 出错怎么办**：
- 每个任务失败后，等 5 分钟重试，最多重试 2 次
- `catchup=False`：电脑关着时错过的月份，不会一个个补跑。开机后只跑最近的一次，反正它取的是最新的数据

**密码在哪？** 文件里没有。Airflow 要用连接 `databricks_default` 时，自己去保险柜取 `airflow/connections/databricks_default`。

### 用了什么技术，为什么
- Airflow 的 TaskFlow 写法：在普通 Python 函数上加 `@task`，它就成了流程图里的一个方框
- Databricks 官方提供的 `DatabricksRunNowOperator`：负责启动 Job 并等它跑完
- **CI 加了一项检查**：DAG 文件能不能被 Airflow 正常读出来。Airflow 最常见的事故就是这个文件写坏了，所有定时运行**悄悄地**停掉，没人发现

### 公司里什么情况怎么做
DAG 文件里只放**顺序和时间**；真正干活的逻辑放在能测试的代码里。这里复用的是 03b 写好、测过的送货代码（[`src/qafi_platform/ingest.py`](../../src/qafi_platform/ingest.py)）。

### 面试怎么说
> "The DAG holds only order and timing: schedule, retries, dependencies. The logic lives in tested library code, and a CI check loads the DAG, so a broken file is caught before it silently stops the schedule."

---

## 第 8 步 · 试运行一次

### 在干什么
打开流程图的开关，让它跑一次，在网页上看着它跑。

### 背景
新流程图默认是暂停的。打开开关以后，有两种情况：
- **这个月 10 号 06:00（UTC）已经过了**：调度器发现"本月这一次还没跑"，马上自己跑
- **还没到**：要手动按一次 **Trigger**（立即运行）

下面这一格自动分辨这两种情况，保证只跑一次。效果和你在网页上打开开关、再点 Trigger 一样。如果你已经在网页上按过 Trigger，它不会再启动一次，而是直接等你按的那一次。

它通过 Airflow 的**网页接口**（REST API）来查和按：程序之间说话用接口，回答是整齐的数据；命令行的输出里还夹着 Airflow 自己的日志，程序读起来容易出错。

In [ ]:
RUN = orchestration.start_dag_run(ROOT, DAG)
print("这次运行:", RUN)

### 在网页上看着它跑（大约 6 分钟）
1. Airflow → `qafi_refresh` → **Grid**：每一列是一次运行，每个小方块是一个任务。浅绿是正在跑，深绿是成功，红色是失败
2. 点 `fetch_new_releases` 的方块 → **Logs**：能看到几行 `skipped ... (already in raw)`。这个月没有新数据，所以什么都不搬
3. 点 `run_lakehouse_job` → **Logs**：有一行 `View run status ... at https://...`，点开就是 Databricks 那边的这次运行，可以看着 6 个方框一个个变绿（07b/07c 之后是 8 个）

然后运行下面这一格，等它跑完：

In [ ]:
orchestration.wait_for_dag_run(ROOT, DAG, RUN)

### 结果怎么看
第一行是 `...: success`，下面 3 个任务都是 `success`。

再从 Databricks 这边确认一遍：

In [ ]:
lakehouse.latest_run(w, JOB)

6 步都是 `SUCCESS`，最后一步 `gold` 就是 05 号的 dbt。

是谁按的"开始"：

In [ ]:
job_id = next(iter(w.jobs.list(name=JOB))).job_id
print("启动这次运行的身份:", next(iter(w.jobs.list_runs(job_id=job_id, limit=1))).creator_user_name, "   qafi-airflow 的卡号:", APP_ID)

两个应该是同一串：是 Airflow 用它自己的卡按的，不是你。

gold 表是这次重建的吗：

In [ ]:
sql(w, f"DESCRIBE HISTORY {CATALOG}.gold.fct_dms_measurement LIMIT 1")[["version", "timestamp", "operation"]]

最新一版的时间就是刚才：说明 gold 表是这次运行重建的。

### 公司里什么情况怎么做
每天早上先看调度系统：
1. 哪一条红了
2. 点进那一步的日志，看出错原因
3. 修好以后**只重跑失败的那一步**：Airflow 里叫 **Clear**，Databricks 里叫 **Repair run**（04 号用过）

### 面试怎么说
> "When a run fails I open the failed task's log, fix the cause and clear just that task. Every task is idempotent, so running it again is always safe."

---

## 第 9 步 · 新版本是怎么自动进来的

### 在干什么
弄清楚下个月 ClinVar 出新版时会发生什么。

### 背景
[`config/sources.toml`](../../config/sources.toml) 里每个数据集都写了 `refresh`：
- **`monthly`**（只有 ClinVar）：Airflow 每月自己去找最新版本
- **`manual`**（UniProt、ProteinGym、AlphaFold）：要人来改版本号，提交审核

**为什么不全部自动**：
- UniProt 换版本，蛋白序列可能会变，05 号的位置换算、07 号的特征都会跟着变
- ProteinGym 换版本，文件的格式可能会变

这些会改变模型的输入，要有人看过再换。ClinVar 是评估用的标签，按月更新是常规操作。

**怎么找最新版本**：不下载，只"敲门问一下"（HTTP 的 HEAD 请求）。先问这个月的文件在不在，不在就问上个月，最多往前问 3 个月；而且永远不会早于 `sources.toml` 里写的版本。

**下个月 10 号会发生什么**：

```
找到 2026-11 → 送货员搬进 datasets/clinvar/2026-11/（旧的 2026-10 还在，不会被覆盖）
            → 提货单里换成 2026-11 的文件
            → 流水线收到参数 clinvar_version=2026-11，重建 ClinVar 的 bronze 表
```

### 执行
问一下：现在每个"每月更新"的数据集，最新的是哪一版：

In [ ]:
ingest.newest_releases(ROOT / "config" / "sources.toml", date.today())

### 结果怎么看
`{'clinvar': '2026-10'}`：现在最新的就是 10 月版。11 月版要到 11 月初才会发布。

### 公司里什么情况怎么做
外部数据的"订阅"是数据工程的日常工作：定时检查、只取新的、按版本存放、记下每次运行用的是哪个版本。

### 面试怎么说
> "Monthly sources refresh automatically into a new versioned folder, and every run records the versions it loaded. Releases that change the model's inputs stay pinned and are raised in a reviewed change."

---

## 第 10 步 · 关掉 Airflow，提交

关掉 Airflow：

In [ ]:
orchestration.stop_airflow(airflow)

关掉以后，它就不会按时跑了。想让它下个月 10 号自己跑，那天电脑要开着，并且运行第 6 步。公司里 Airflow 是一直开着的服务，没有这个问题。

### 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 06 orchestration with Lakeflow Jobs, Airflow and Secrets Manager") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼：多了一项 `airflow`（检查 DAG 文件能不能读出来），应该是绿的。

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 编排 | 流程图、定时、重试、重跑安全 | 数据按时更新，失败能自己恢复 |
| 2 两个调度员 | Airflow + Lakeflow Jobs | 跨系统的大顺序 + Databricks 里的小顺序 |
| 3 保险柜 | AWS Secrets Manager + Terraform | 密码不在代码里，也不在状态文件里 |
| 4 门禁卡 | 机器身份 + OAuth | 程序有自己的身份，只给够用的权限 |
| 5 流水线 | Lakeflow Job + dbt 任务 | raw 到 gold 一口气跑完，失败发邮件 |
| 6–8 Airflow | standalone、DAG、Grid、日志 | 按时跑，出错能看到在哪一步 |
| 9 新版本 | `refresh` + HEAD 请求 | 外部数据自动进来，并记下版本 |

**一句话**：从现在起，训练数据会每个月自己更新，从 ClinVar 新版本进 S3，一直到 gold 表重建；整个过程没有一个人需要按按钮，也没有一个密码写在代码里。

## 面试题

1. **Airflow or Databricks Jobs: which would you use?**
   > "Lakeflow Jobs for the steps inside Databricks: it handles compute, retries and repair natively. Airflow when a pipeline spans systems, as here: new data lands on AWS first, then the Databricks job runs. One of them owns the schedule, never both."

2. **How do your pipelines authenticate?**
   > "Each automation has its own identity: CI uses GitHub OIDC into AWS, and Airflow uses a Databricks service principal with OAuth machine-to-machine. The only long-lived secret sits in Secrets Manager and rotates; nothing is in code or in the Terraform state."

3. **A scheduled run failed at 3 a.m. What happens next?**
   > "Transient steps retry by themselves; if it still fails, the owner gets an email. In the morning I read the failed task's log, fix the cause and re-run only that task, which is safe because every task is idempotent."

4. **How do you bring in new versions of external data?**
   > "Monthly sources are checked on a schedule and land in a new versioned folder, never overwriting the old one, and each run records the versions it used. Releases that change model inputs are pinned and updated in a reviewed change."

**下一章 07 · 特征和实验**：从 raw 算模型的特征（AlphaFold 结构等，位置用 05 号换算好的 `uniprot_position`），每次实验记进 MLflow。